In [ ]:
!nvidia-smi

Tue Sep 15 13:40:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
MODELS = {
    "krikri":  "ilsp/Llama-Krikri-8B-Instruct",
    "meltemi": "ilsp/Meltemi-7B-Instruct-v1.5",
    "llama":   "meta-llama/Llama-3.1-8B-Instruct",
}

LOAD_IN_4BIT = True    # same as the inference run


print("4-bit:", LOAD_IN_4BIT)

4-bit: True


In [ ]:
import gc
import json

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_model(hf_id):
    """Load one model and its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    settings = {"device_map": "auto", "torch_dtype": torch.float16}
    if LOAD_IN_4BIT:
        settings["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(hf_id, **settings)
    model.eval()
    return model, tokenizer


def measure_surprise(model, tokenizer, text):
    """How surprised the model is by this text.

    Returns (average loss per token, number of tokens).
    Lower loss means the text was easier to predict.
    """
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_tokens = inputs["input_ids"].shape[1]

    # passing labels makes the model predict each token from the ones before it
    # and report how wrong it was. No text is generated.
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return outputs.loss.item(), n_tokens


def read_texts(filename, level):
    """Read one input file into a list of rows."""
    texts = []

    with open(filename, encoding="utf-8") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            texts.append({
                "text_id": row["text_id"],
                "level": level,
                "tasks": row["tasks"],
                "pharasiot_text": row["pharasiot_text"],
                "greek_text": row["greek_text"],
            })

    return texts


def run(model_name):
    """Measure surprise for every text, in both varieties, and save the results."""
    print("loading", model_name)
    model, tokenizer = load_model(MODELS[model_name])

    results = []

    for level, filename in INPUT_FILES.items():
        texts = read_texts(filename, level)

        for row in tqdm(texts, desc="{} {}".format(model_name, level)):
            for variety in ["pharasiot", "greek"]:
                text = row[variety + "_text"]
                loss, n_tokens = measure_surprise(model, tokenizer, text)

                results.append({
                    "text_id": row["text_id"],
                    "level": row["level"],
                    "tasks": row["tasks"],
                    "model": model_name,
                    "variety": variety,
                    "n_tokens": n_tokens,
                    "loss": round(loss, 4),
                })

    path = "crossentropy/{}.jsonl".format(model_name)
    with open(path, "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("wrote {}  ({} rows)".format(path, len(results)))

    # free the GPU before the next model
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import os
os.makedirs("crossentropy", exist_ok=True)

In [ ]:
import csv
from google.colab import files

uploaded = files.upload()

with open("crossentropy_targets.tsv", encoding="utf-8") as f:
    targets = list(csv.DictReader(f, delimiter="\t"))

print(len(targets), "target words")

Saving crossentropy_targets.tsv to crossentropy_targets.tsv
40 target words


In [ ]:
import torch.nn.functional as F


def token_losses(model, tokenizer, text):
    """Loss for every token, plus where each token sits in the text."""
    encoded = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    # the model predicts token i+1 from token i, so line them up
    predictions = outputs.logits[0, :-1]
    actual = encoded["input_ids"][0, 1:]

    losses = F.cross_entropy(predictions, actual, reduction="none").tolist()

    # losses[i] belongs to the token at offsets[i + 1]
    return losses, offsets[1:]


def score_target(model, tokenizer, sentence, start, end):
    """Mean loss over the tokens inside the target word.

    Returns (mean loss, how many tokens the word used).
    """
    losses, offsets = token_losses(model, tokenizer, sentence)

    inside = []
    for i in range(len(losses)):
        token_start, token_end = offsets[i]
        # a token counts if it overlaps the word. Tokenizers often attach the
        # leading space, so the span can begin one character early.
        if token_end > start and token_start < end:
            inside.append(losses[i])

    if not inside:
        return None, 0

    return sum(inside) / len(inside), len(inside)


def run_targets(model_name):
    """Score every target word and save the results."""
    print("loading", model_name)
    model, tokenizer = load_model(MODELS[model_name])

    results = []

    for row in tqdm(targets, desc=model_name):
        start = int(row["char_start"])
        end = int(row["char_end"])

        loss, n_tokens = score_target(model, tokenizer, row["sentence"], start, end)
        sentence_loss, sentence_tokens = measure_surprise(
            model, tokenizer, row["sentence"])

        results.append({
            "item_id": row["item_id"],
            "target_word": row["target_word"],
            "gloss": row["gloss"],
            "model": model_name,
            "word_loss": None if loss is None else round(loss, 4),
            "word_tokens": n_tokens,
            "sentence_loss": round(sentence_loss, 4),
            "sentence_tokens": sentence_tokens,
        })

    path = "crossentropy/targets_{}.jsonl".format(model_name)
    with open(path, "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("wrote {}  ({} rows)".format(path, len(results)))

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
run_targets("krikri")

loading krikri


config.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 19.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

krikri:   0%|          | 0/40 [00:00<?, ?it/s]

wrote crossentropy/targets_krikri.jsonl  (40 rows)


In [ ]:
run_targets("meltemi")

loading meltemi


config.json:   0%|          | 0.00/713 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.97M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.18MB            

tokenizer.model: downloading bytes:           |  0.00B            

... (5 outputs left out)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

meltemi:   0%|          | 0/40 [00:00<?, ?it/s]

wrote crossentropy/targets_meltemi.jsonl  (40 rows)


In [ ]:
run_targets("llama")

loading llama


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

llama:   0%|          | 0/40 [00:00<?, ?it/s]

wrote crossentropy/targets_llama.jsonl  (40 rows)


In [ ]:
for name in sorted(os.listdir("crossentropy")):
    if not name.startswith("targets_"):
        continue

    rows = [json.loads(line) for line in open("crossentropy/" + name, encoding="utf-8")]
    unmatched = [r for r in rows if r["word_loss"] is None]
    scored = [r for r in rows if r["word_loss"] is not None]

    print(name, " n =", len(rows), " unmatched =", len(unmatched))
    print("   mean word loss     {:.3f}".format(
        sum(r["word_loss"] for r in scored) / len(scored)))
    print("   mean sentence loss {:.3f}".format(
        sum(r["sentence_loss"] for r in scored) / len(scored)))
    print("   mean tokens per target word {:.1f}".format(
        sum(r["word_tokens"] for r in scored) / len(scored)))

    hardest = sorted(scored, key=lambda r: -r["word_loss"])[:5]
    print("   most surprising words:")
    for r in hardest:
        print("      {:16} {:.2f}  ({} tokens)".format(
            r["target_word"], r["word_loss"], r["word_tokens"]))
    print()

targets_krikri.jsonl  n = 40  unmatched = 0
   mean word loss     7.394
   mean sentence loss 6.748
   mean tokens per target word 3.3
   most surprising words:
      ασλάνος          10.22  (3 tokens)
      Σηκώθην          10.16  (3 tokens)
      çαρέν            10.12  (3 tokens)
      Ποίjεν           10.09  (4 tokens)
      αν               9.70  (1 tokens)

targets_llama.jsonl  n = 40  unmatched = 0
   mean word loss     6.635
   mean sentence loss 6.183
   mean tokens per target word 4.3
   most surprising words:
      çαρέν            9.97  (3 tokens)
      Ποίjεν           9.48  (5 tokens)
      Μαρgάωσανε       9.43  (7 tokens)
      Χορλατίζει       9.28  (5 tokens)
      Αωπός            8.45  (4 tokens)

targets_meltemi.jsonl  n = 40  unmatched = 0
   mean word loss     7.194
   mean sentence loss 6.399
   mean tokens per target word 3.2
   most surprising words:
      κετσιντήσετε     11.22  (3 tokens)
      Σωρεύτανε        10.27  (3 tokens)
      αν               8.96  

In [ ]:
!zip -qr crossentropy_results.zip crossentropy

from google.colab import files

files.download("crossentropy_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>